# QML 강의 - 2일차

## HAM10000 전처리와 QGAN 학습

- 목표: HAM10000 DF 이미지를 segmentation 전처리하고, QGAN(양자 생성자 + 고전 판별자)을 학습해 생성 결과를 지표 4종으로 평가함
- 순서: 환경 세팅 → EDA → segmentation → 회로 구성 → QGAN 학습 → 생성·평가 → Drive 저장
- 회로 규모: sub-generator 4개 × 7큐빗(데이터 6 + ancilla 1), depth 10. 논문(sub-generator 16개, 데이터 7 + ancilla 1)보다 줄임 - Colab CPU에서 수업 시간 안에 학습하기 위함
- 개념 설명은 2일차 설명 자료(`practice/day2_slides/`)에 있음

## 0. Google Colab 시작하기

- Google 계정과 브라우저만 있으면 됨(설치 불필요). 계정이 없으면 https://accounts.google.com/signup 에서 만들고, 학교·회사 계정이 막혀 있으면 개인 Gmail을 씀
- 저장소(https://github.com/hyunhp/skin_disease_qml) 첫 화면의 **Open in Colab**으로 열고, 바로 `파일(File) → Drive에 사본 저장`을 실행함. 사본이 없으면 실행 결과가 저장되지 않음
- "이 노트북은 Google에서 작성하지 않았습니다" 경고가 뜨면 `무시하고 계속(Run anyway)`
- 런타임은 기본 CPU 그대로 둠(GPU로 바꿔도 빨라지지 않음)

| 조작 | 방법 |
|---|---|
| 셀 실행 | Shift + Enter 또는 셀 왼쪽 ▶ |
| 실행 중단 | 셀 왼쪽 ■ 또는 `런타임 → 실행 중단` |
| 초기화 | `런타임 → 세션 다시 시작` 후 1번 섹션부터 다시 실행 |
| 파일 확인 | 왼쪽 사이드바 📁 |

- 셀은 위에서부터 순서대로 실행함. 건너뛰면 `NameError`가 남
- 유휴 약 90분(최대 12시간)이면 런타임이 끊기고 설치·데이터·변수가 사라짐. 다시 연결해 1번 섹션부터 실행함. 결과물은 7번 섹션에서 Drive에 저장함
- 오류가 나면 메시지 마지막 줄을 먼저 확인함. 대부분 앞 셀 미실행이나 연결 끊김이 원인임

## 1. 환경 세팅

아래 셀을 순서대로 실행함. 전부 실행되면 다음 섹션으로 진행함.

In [ ]:
# 필요 라이브러리 설치
!pip install -q pennylane pennylane-lightning scikit-fuzzy kagglehub koreanize-matplotlib
print("설치 완료")

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import torch
import pennylane as qml
import skfuzzy as fuzz
from sklearn.cluster import KMeans

try:
    import koreanize_matplotlib  # 나눔고딕 폰트 등록 + 기본 폰트 지정
except Exception as e:
    print("한글 폰트 설정 실패(그래프 한글만 깨짐, 진행에는 지장 없음):", e)

print(f"torch: {torch.__version__}")
print(f"pennylane: {qml.__version__}")
print(f"opencv: {cv2.__version__}")

# 동작 확인 - |0⟩에 Hadamard를 걸면 ⟨Z⟩ = 0 이 나와야 정상
dev = qml.device("lightning.qubit", wires=1)
@qml.qnode(dev)
def _sanity_check():
    qml.Hadamard(wires=0)
    return qml.expval(qml.PauliZ(0))

print(f"PennyLane 동작 확인: {_sanity_check()}")

## 2. HAM10000 데이터셋 로드 + EDA

HAM10000: 피부경검사 이미지 10,015장, 7개 클래스.

- `kagglehub`로 Colab 런타임에 직접 내려받음(약 6GB, 개인 PC·Drive에는 저장되지 않음). 같은 런타임에서 재실행하면 캐시를 씀
- 경로가 `/kaggle/input/...`으로 나오면 Kaggle의 Colab 캐시를 바로 연결한 것으로, 정상임
- 공개 데이터셋이라 로그인 없이 받음. 아래 로그인 셀은 건너뛰고, 다운로드 셀에서 인증 오류(401/403)가 날 때만 씀:
  1. https://www.kaggle.com/settings 에서 API 토큰 발급(아래 캡처)
  2. 로그인 셀의 `# kagglehub.login()` 주석을 해제하고 실행 → 토큰 붙여넣기 → Login
  3. "Kaggle credentials successfully validated"를 확인한 뒤 다운로드 셀을 다시 실행함
- 토큰은 비밀번호처럼 다룸. 노트북에 적거나 공유하지 않음

### (참고) Kaggle API 토큰 발급 화면

**① 우측 상단 프로필 → `Settings`**

<img src="https://raw.githubusercontent.com/hyunhp/skin_disease_qml/main/practice/docs/image/kaggle_1_account_menu.png" width="320">

**② `API Tokens` 탭 → `Generate New Token`**

<img src="https://raw.githubusercontent.com/hyunhp/skin_disease_qml/main/practice/docs/image/kaggle_2_api_tokens_tab.png" width="760">

**③ 토큰 이름(예: `qml_practice`)을 적고 `Generate`**

<img src="https://raw.githubusercontent.com/hyunhp/skin_disease_qml/main/practice/docs/image/kaggle_3_generate_token.png" width="760">

**④ 표시된 토큰을 바로 복사** - 창을 닫으면 다시 볼 수 없음(못 받았으면 ②부터 새로 발급)

<img src="https://raw.githubusercontent.com/hyunhp/skin_disease_qml/main/practice/docs/image/kaggle_4_token_shown.png" width="420">

사용: `kagglehub.login()`에 붙여넣거나, `os.environ["KAGGLE_API_TOKEN"] = "복사한 토큰"`을 실행한 뒤 다운로드 셀을 다시 실행함.

In [ ]:
import kagglehub

# # (선택) 다운로드에서 인증 오류가 날 때만 주석 해제 후 실행
# # 토큰 붙여넣기 → Login → 성공 메시지 확인 후 다운로드 셀 재실행
# kagglehub.login()

# # (대안1) 신형 API 토큰(KGAT_로 시작)을 환경변수로 지정
# os.environ["KAGGLE_API_TOKEN"] = "복사한 토큰"

# # (대안2) 구형 kaggle.json 방식 자격증명
# os.environ["KAGGLE_USERNAME"] = "본인 Kaggle 아이디"
# os.environ["KAGGLE_KEY"] = "본인 API key"

In [ ]:
import kagglehub

# 약 6GB (캐시가 있으면 재사용). 경로가 /kaggle/input/...이면 Colab 캐시 연결로 정상
try:
    DATA_DIR = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
except Exception as e:
    raise RuntimeError("다운로드 실패 - 네트워크 확인 후 재실행. 인증 오류면 위 (선택) 로그인 셀 실행 후 재시도") from e
print("데이터셋 경로:", DATA_DIR)
print("포함 항목:", sorted(os.listdir(DATA_DIR)))

#### 전체 흐름

```
원본 이미지 (600×450 RGB, 10,015장)
   │ ① EDA: DF vs NV 이진분류로 범위 축소
   ▼
DF 115장 / NV 6,705장 (약 1:58)
   │ ② segmentation: grayscale → 64×64 → K-means → Fuzzy C-means → 병변만 남김
   ▼
병변만 남은 64×64
   │ ③ 축소: 64×64 → 16×16, 0 ~ 1 정규화
   ▼
256픽셀 = sub-generator 4개 × 64픽셀(2⁶) → 회로 하나는 데이터 6 + ancilla 1 = 7큐빗
   ▼
QGAN 학습 → 생성 → 지표 4종 평가
```

- EDA: 논문(Andra et al.)과 같이 가장 적은 DF와 가장 많은 NV만 씀
- segmentation: 논문 Table 10에서 raw 이미지 증강은 분류 성능을 떨어뜨렸고 segmentation 이미지에서만 개선됨. 16×16까지 줄이기 전에 배경을 지워야 병변 정보가 남음
- 한계: 가장 어두운 cluster를 병변으로 보므로, 이미지 모서리의 검은 테두리가 병변으로 잡힐 수 있음
- `IMG_SIZE = 8`로 바꾸면 더 작게 돌릴 수 있지만, 3일차 자산(16×16)과 비교할 수 없음

In [ ]:
# 이미지별 진단명(dx). 파일명 = image_id
metadata = pd.read_csv(f"{DATA_DIR}/HAM10000_metadata.csv")
print(f"전체 이미지 수: {len(metadata)}")
metadata.head()

In [ ]:
# 클래스별 이미지 수 (논문 Fig.12)
class_names_kr = {"akiec": "광선각화증", "bcc": "기저세포암", "bkl": "양성각화증",
                   "df": "피부섬유종", "mel": "흑색종", "nv": "모반(정상)", "vasc": "혈관병변"}
counts = metadata["dx"].value_counts()  # 많은 순
plt.figure(figsize=(8, 4))
plt.bar([class_names_kr[c] for c in counts.index], counts.values)
plt.ylabel("이미지 수"); plt.title("HAM10000 클래스별 분포")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()
print(counts)
print(f"\nDF(피부섬유종): {counts.get('df', 0)}장 vs NV(모반): {counts.get('nv', 0)}장",
      f"- 불균형 비율 약 1:{counts.get('nv',0)//max(1,counts.get('df',1))}")

In [ ]:
# 클래스별 첫 이미지
def find_image_path(image_id):
    # part_1·part_2 두 폴더를 모두 확인
    for sub in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
        p = f"{DATA_DIR}/{sub}/{image_id}.jpg"
        if os.path.exists(p):
            return p
    return None

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, cls in zip(axes.flat, class_names_kr.keys()):
    row = metadata[metadata["dx"] == cls].iloc[0]
    path = find_image_path(row["image_id"])
    if path:
        # BGR → RGB (표시용)
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        ax.imshow(img); ax.set_title(f"{cls} ({class_names_kr[cls]})"); ax.axis("off")
axes.flat[-1].axis("off")  # 8칸 중 마지막은 빈칸
plt.tight_layout(); plt.show()

## 3. 전처리 실습 - segmentation

논문 Section 3.1 방식: grayscale → K-means로 대략 나눔 → Fuzzy C-means로 정제 → 가장 어두운 cluster(병변)만 남김.

### segmentation 함수 요약

- `KMeans(n_clusters=4, n_init=4)`: 밝기만으로 4개 cluster로 나눔. 초기화(k-means++)를 4번 시도해 가장 좋은 결과를 씀
- `fuzz.cluster.cmeans(..., m=2.0, init=u_init)`: 픽셀마다 cluster별 소속도를 나눠 가짐. K-means 결과로 초기 소속도를 만들어 빠르고 매번 같은 결과로 수렴함
- `cntr`(cluster 중심 밝기) 중 가장 어두운 cluster를 병변으로 보고, `u`(소속도)의 argmax로 픽셀을 배정함
- `cv2.bitwise_and(gray, gray, mask=mask)`: 병변은 원래 밝기, 배경은 0
- 인자·출력 설명은 2일차 설명 자료 PART 1

In [ ]:
def segment_fcm_kmeans(img_bgr, n_clusters=4, resize_to=None):
    """HAM10000 QGAN 논문 Section 3.1 - K-means+Fuzzy C-means hybrid segmentation."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)  # 밝기 1채널로 clustering
    if resize_to is not None:
        gray = cv2.resize(gray, resize_to, interpolation=cv2.INTER_AREA)
    h, w = gray.shape
    pixels = gray.reshape(-1, 1).astype(np.float32)  # (픽셀 수, 1): 밝기만 사용

    # 1단계: K-means 사전 clustering (k-means++ 초기화 4회 중 최선)
    km = KMeans(n_clusters=n_clusters, n_init=4, random_state=0).fit(pixels)
    # FCM 초기 소속도 = K-means 중심까지 거리의 역제곱
    dist = np.abs(pixels - km.cluster_centers_.reshape(1, -1)) + 1e-6  # 분모 0 방지
    u_init = 1.0 / dist ** 2
    u_init = (u_init / u_init.sum(axis=1, keepdims=True)).T  # (cluster 수, 픽셀 수)

    # 2단계: Fuzzy C-means로 정제
    cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
        pixels.T, c=n_clusters, m=2.0, error=1e-4, maxiter=100, init=u_init)
    membership = np.argmax(u, axis=0)  # 소속도 최대 cluster로 라벨
    lesion_cluster = np.argmin(cntr.flatten())  # 가장 어두운 cluster = 병변
    # 병변 픽셀만 255
    mask = (membership == lesion_cluster).reshape(h, w).astype(np.uint8) * 255

    # 3단계: 병변은 원래 밝기, 배경은 0
    return cv2.bitwise_and(gray, gray, mask=mask)

print("함수 정의 완료")

In [ ]:
# DF 첫 이미지로 확인. 64x64로 줄인 뒤 clustering (원본은 FCM이 느림)
df_row = metadata[metadata["dx"] == "df"].iloc[0]
img_bgr = cv2.imread(find_image_path(df_row["image_id"]))
segmented = segment_fcm_kmeans(img_bgr, n_clusters=4, resize_to=(64, 64))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)); axes[0].set_title("원본"); axes[0].axis("off")
axes[1].imshow(segmented, cmap="gray"); axes[1].set_title("segmentation 결과"); axes[1].axis("off")
plt.tight_layout(); plt.show()

### 해상도 축소와 보간법

64×64 → 16×16처럼 크게 줄일 때는 `cv2.INTER_AREA`(출력 픽셀이 덮는 영역의 평균)를 씀. 기본값 `INTER_LINEAR`는 주변 2×2만 참조해 작은 병변 정보가 사라지거나 앨리어싱이 생김. 저장소 자산(`practice/assets/*.npy`)도 같은 `INTER_AREA`로 만듦.

In [ ]:
# 16x16 = 256픽셀 = sub-generator 4개 × 64픽셀
IMG_SIZE = 16        # 저장소 자산과 같아야 함
N_GENERATORS = 4     # patch 64픽셀씩 담당

def resize_for_encoding(img_gray, size):
    # 큰 폭 축소라 INTER_AREA (저장소 자산과 같은 조건)
    return cv2.resize(img_gray, size, interpolation=cv2.INTER_AREA)

# 64x64 → 16x16
small = resize_for_encoding(segmented, (IMG_SIZE, IMG_SIZE))
plt.figure(figsize=(3, 3))
plt.imshow(small, cmap="gray"); plt.title(f"{IMG_SIZE}x{IMG_SIZE} 축소본(양자 encoding용)"); plt.axis("off")
plt.show()

patch_px = IMG_SIZE * IMG_SIZE // N_GENERATORS  # sub-generator 1개가 맡는 픽셀 수
n_data_q = int(np.log2(patch_px))  # 2^n = patch 픽셀 수
print(f"전체 픽셀 수: {IMG_SIZE*IMG_SIZE}")
print(f"sub-generator {N_GENERATORS}개 -> patch당 {patch_px}픽셀 -> 데이터 큐빗 {n_data_q}개 (2^{n_data_q}={patch_px})")
print("참고: 논문은 sub-generator 16개 x (데이터 7 + ancilla 1)큐빗")

In [ ]:
# 클래스 하나를 전처리해 (장수, 16, 16) 배열로 반환
def preprocess_class(metadata, dx_class, img_size, max_n=None):
    rows = metadata[metadata["dx"] == dx_class]
    if max_n:
        rows = rows.head(max_n)
    imgs = []
    for _, row in rows.iterrows():
        path = find_image_path(row["image_id"])
        if path is None:
            continue
        bgr = cv2.imread(path)
        # 64x64 segmentation → 16x16 → 0~1
        seg = segment_fcm_kmeans(bgr, n_clusters=4, resize_to=(64, 64))
        imgs.append(resize_for_encoding(seg, (img_size, img_size)).astype(np.float32) / 255.0)
    return np.stack(imgs) if imgs else np.zeros((0, img_size, img_size), dtype=np.float32)

# 5장만 직접 처리(1장 수 초). 학습에는 다음 섹션의 저장소 자산(DF 115장)을 씀
df_demo = preprocess_class(metadata, "df", IMG_SIZE, max_n=5)
nv_demo = preprocess_class(metadata, "nv", IMG_SIZE, max_n=5)
print(f"직접 전처리한 샘플 - DF {df_demo.shape}, NV {nv_demo.shape}")

fig, axes = plt.subplots(2, 5, figsize=(11, 4.6))
for j in range(5):
    axes[0, j].imshow(df_demo[j], cmap="gray"); axes[0, j].set_title(f"DF {j}", fontsize=9); axes[0, j].axis("off")
    axes[1, j].imshow(nv_demo[j], cmap="gray"); axes[1, j].set_title(f"NV {j}", fontsize=9); axes[1, j].axis("off")
plt.tight_layout(); plt.show()

## 4. encoding + 회로 구성

sub-generator 하나를 PennyLane으로 구성하고 학습 전 출력을 확인함.

- 구조: 노이즈를 RY로 encoding → [학습 RY + 인접 CZ] × 10 → 7큐빗 전체 측정
- post-selection: ancilla qubit = 0인 확률 64개만 patch로 씀. 데이터 큐빗만 측정하면 64개의 합이 항상 1이라 배경 patch(합 ≈ 0)를 표현할 수 없음

In [ ]:
import math

N_DATA_QUBITS = int(np.log2(IMG_SIZE * IMG_SIZE // N_GENERATORS))  # 256/4=64 -> 6큐빗
N_ANCILLA = 1  # post-selection용 ancilla qubit (위 마크다운 참고)
N_QUBITS = N_DATA_QUBITS + N_ANCILLA  # 7큐빗 → 상태벡터 2^7 = 128차원
Q_DEPTH = 10  # 논문과 동일

dev = qml.device("lightning.qubit", wires=N_QUBITS)  # 출력 확인용 (학습은 다음 섹션)

@qml.qnode(dev)
def sub_generator_circuit(noise, weights):
    # 1) 노이즈를 RY 각도로 encoding (샘플마다 다른 이미지의 원천)
    for i in range(N_QUBITS):
        qml.RY(noise[i], wires=i)
    # 2) 변분 layer: RY 회전각(weights)이 학습 대상
    for layer in range(Q_DEPTH):
        for i in range(N_QUBITS):
            qml.RY(weights[layer, i], wires=i)
        # 인접 큐빗 CZ로 얽힘
        for i in range(N_QUBITS - 1):
            qml.CZ(wires=[i, i + 1])
    # 3) 7큐빗 전체 측정 → 확률 128개
    return qml.probs(wires=range(N_QUBITS))

print(f"큐빗 수: {N_QUBITS} (데이터 {N_DATA_QUBITS} + ancilla {N_ANCILLA}), depth={Q_DEPTH}")

# 학습 전 랜덤 파라미터로 출력만 확인
noise = np.random.rand(N_QUBITS) * math.pi
weights = np.random.rand(Q_DEPTH, N_QUBITS) * math.pi
probs_all = sub_generator_circuit(noise, weights)
# post-selection: ancilla qubit(마지막 wire = 최하위 비트)이 0인 짝수 인덱스만
output = probs_all[::2]
print(f"전체 측정 확률 개수: {probs_all.shape} (=2^{N_QUBITS}) / 전체 합: {float(np.sum(probs_all)):.6f} (1이어야 정상)")
print(f"ancilla qubit=0 항목만 선택한 출력 shape: {output.shape} (=2^{N_DATA_QUBITS}={IMG_SIZE*IMG_SIZE//N_GENERATORS}이어야 함)")
print(f"선택된 확률의 합 (0~1, 학습되는 '밝기 총합'): {float(np.sum(output)):.4f}")

In [ ]:
# sub-generator 1개 = patch 1개(이미지의 1/4). 학습 전이라 랜덤 패턴이 정상
patch_side = int(np.sqrt(len(output) * N_GENERATORS))  # 이미지 한 변 16 → patch는 16열 x 4행
plt.figure(figsize=(3, 2))
plt.imshow(np.array(output).reshape(-1, patch_side), cmap="gray")
plt.title("학습 전 sub-generator 출력 patch (랜덤 - 정상)")
plt.axis("off")
plt.show()
print(f"patch 1개 = {len(output)}픽셀 = {IMG_SIZE}x{IMG_SIZE} 이미지의 1/{N_GENERATORS}")
print(f"다음 섹션에서 이 회로 {N_GENERATORS}개를 묶어 학습시키면 병변 이미지 패턴을 생성하게 됨")

## 5. QGAN 학습

sub-generator 4개를 묶은 생성자와 논문 Table 4의 고전 판별자(5층 FC)를 경쟁시켜 학습함.

- 데이터: 저장소의 전처리 자산(DF 115장). 직접 전처리한 `df_demo` 5장으로는 학습되지 않음
- 척도: 생성자 출력이 확률이므로 실제 이미지도 장당 픽셀 합이 1이 되게 바꿔 판별자에 넣음. 평가할 때는 `to_unit_scale(x, "prob")`로 0 ~ 1로 되돌림
- 한계: 실제 이미지는 장당 합이 1, 생성 이미지는 patch별 합이라 장당 0 ~ 4(Colab 실측 약 1.63). 판별자가 형태 대신 밝기 총합으로 구분할 여지가 있음. 지표는 이미지별 min-max로 다시 맞춰 계산하므로 직접 영향은 없음. README·강사 checkpoint 수치도 이 방식으로 학습한 결과임

In [ ]:
# 강의 저장소 내려받기 (전처리 자산 + 지표 모듈 사용)
import sys, subprocess, os

REPO_DIR = "/content/skin_disease_qml"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/hyunhp/skin_disease_qml.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)  # 재실행 시 최신 자산으로 갱신

ASSET_DIR = f"{REPO_DIR}/practice/assets"
sys.path.insert(0, f"{REPO_DIR}/practice")      # gen_metrics 모듈 사용
import gen_metrics as gm

# 같은 pipeline으로 미리 처리한 DF 115장·NV 345장 (0~1)
df_full = np.load(f"{ASSET_DIR}/df_preprocessed.npy")
nv_full = np.load(f"{ASSET_DIR}/nv_preprocessed.npy")
print(f"학습용 DF {df_full.shape}, 비교용 NV {nv_full.shape} (값 범위 {df_full.min():.2f}~{df_full.max():.2f})")
assert df_full.shape[1] == IMG_SIZE, f"자산 해상도가 IMG_SIZE({IMG_SIZE})와 다름"

In [ ]:
import torch
import torch.nn as nn

class PatchQuantumGenerator(nn.Module):
    """sub-generator N개가 patch를 나눠 만들고 이어붙여 이미지 한 장을 생성함."""

    def __init__(self, n_generators, n_data_qubits, n_ancillas=1, q_depth=10):
        super().__init__()
        self.n_generators = n_generators
        self.n_qubits = n_data_qubits + n_ancillas
        self.patch_dim = 2 ** n_data_qubits   # patch 하나의 픽셀 수 (64)
        self.anc_stride = 2 ** n_ancillas     # ancilla qubit=0 항목을 고르는 간격 (2 → [::2])
        # 4번 섹션과 같은 회로. PyTorch 자동미분을 위해 default.qubit + backprop
        q_dev = qml.device("default.qubit", wires=self.n_qubits)
        n_q, depth = self.n_qubits, q_depth

        @qml.qnode(q_dev, interface="torch", diff_method="backprop")
        def circuit(noise, weights):
            for i in range(n_q):
                qml.RY(noise[i], wires=i)
            for layer in range(depth):
                for i in range(n_q):
                    qml.RY(weights[layer, i], wires=i)
                for i in range(n_q - 1):
                    qml.CZ(wires=[i, i + 1])
            return qml.probs(wires=range(n_q))

        self.circuit = circuit
        # sub-generator마다 독립된 회전각 (depth x 큐빗) = 70개, 4개 합쳐 280개
        self.q_params = nn.ParameterList([
            nn.Parameter(torch.rand(q_depth, self.n_qubits) * math.pi) for _ in range(n_generators)
        ])

    def forward(self, batch_size):
        # sub-generator별 patch를 이어붙여 (batch, 256)
        patches = []
        for g in range(self.n_generators):
            w = self.q_params[g]
            # 샘플마다 새 노이즈로 회로 1회 실행, ancilla qubit=0 항목만 사용
            rows = [self.circuit(torch.rand(self.n_qubits) * math.pi, w)[::self.anc_stride]
                    for _ in range(batch_size)]
            patches.append(torch.stack(rows))
        return torch.cat(patches, dim=1).float()

    @property
    def output_dim(self):
        return self.n_generators * self.patch_dim


class ClassicalDiscriminator(nn.Module):
    """논문 Table 4 그대로 - 5층 완전연결, ReLU, sigmoid 출력."""

    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 8), nn.ReLU(),
            nn.Linear(8, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


gen = PatchQuantumGenerator(N_GENERATORS, N_DATA_QUBITS, N_ANCILLA, Q_DEPTH)
disc = ClassicalDiscriminator(input_dim=gen.output_dim)
print(f"생성자 출력 차원: {gen.output_dim} (sub-generator {gen.n_generators} x patch {gen.patch_dim})")
print(f"학습 파라미터 수 - 생성자 {sum(p.numel() for p in gen.parameters())}개 / "
      f"판별자 {sum(p.numel() for p in disc.parameters())}개")

### 진행 로그 읽는 법

다음 셀은 Colab 기본 CPU에서 400 step 약 35분 걸림(실측). 로그는 20 step마다 출력됨.

| 값 | 의미 |
|---|---|
| `loss_d` ≈ 2 ln 2 ≈ 1.386 | 판별자가 진짜와 가짜를 구분하지 못함(경쟁 유지) |
| `loss_g` ≈ ln 2 ≈ 0.693 | 판별자가 생성 이미지를 반반으로 판정 |
| `loss_g` = 0.85 | D(G(z)) ≈ exp(−0.85) ≈ 0.43 |

- `loss_d`가 0으로 떨어지면서 `loss_g`가 커지면 판별자가 이기는 중임(README "학습 안정성")
- 한 step의 계산 과정, 코드 해설, parameter-shift와의 비용 비교는 2일차 설명 자료 PART 2 ~ 3

In [ ]:
import time

# 실제 이미지를 확률 척도로 (장당 합 = 1). 생성 이미지와 척도가 다름 (위 '한계' 참고)
real_flat = torch.tensor(df_full.reshape(len(df_full), -1), dtype=torch.float32)
real_flat = real_flat / real_flat.sum(dim=1, keepdim=True).clamp(min=1e-6)

bce = nn.BCELoss()  # 판별자 출력(진짜일 확률) vs 라벨(1/0)
# 파라미터 수 차이(생성자 280, 판별자 19,201)로 학습률을 다르게 둠
opt_g = torch.optim.Adam(gen.parameters(), lr=0.05)
opt_d = torch.optim.Adam(disc.parameters(), lr=0.001)

STEPS = 400       # Colab CPU 실측 약 35분. 시간이 부족하면 300으로 줄임
BATCH = 8
hist_g, hist_d = [], []

t0 = time.time()
for step in range(STEPS):
    # 실제 DF에서 BATCH장 무작위 추출(복원추출)
    batch = real_flat[torch.randint(0, len(real_flat), (BATCH,))]
    real_lbl = torch.ones(BATCH, 1)
    fake_lbl = torch.zeros(BATCH, 1)

    # 판별자: 실제→1, 생성→0 (detach로 생성자 기울기 제외)
    opt_d.zero_grad()
    loss_d = bce(disc(batch), real_lbl) + bce(disc(gen(BATCH).detach()), fake_lbl)
    loss_d.backward(); opt_d.step()

    # 생성자: 생성 이미지를 1로 판정받도록 학습 (양자회로까지 역전파)
    opt_g.zero_grad()
    loss_g = bce(disc(gen(BATCH)), real_lbl)
    loss_g.backward(); opt_g.step()

    hist_g.append(loss_g.item()); hist_d.append(loss_d.item())
    if (step + 1) % 20 == 0:
        el = time.time() - t0
        print(f"step {step+1}/{STEPS}  loss_g={loss_g.item():.4f}  loss_d={loss_d.item():.4f}  "
              f"경과 {el/60:.1f}분  남은 예상 {(STEPS-step-1)*el/(step+1)/60:.1f}분", flush=True)

print(f"\n학습 완료 - 총 {(time.time()-t0)/60:.1f}분")

### 손실곡선 읽는 법

GAN 손실은 단조 감소하지 않음. 손실이 균형처럼 보여도 품질은 낮을 수 있어, 다음 섹션의 지표로 따로 평가함.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
gm.plot_loss_curves(hist_g, hist_d, ax=ax)
ax.set_title("QGAN 학습 손실곡선")
plt.tight_layout(); plt.show()

## 6. 생성 + 생성품질 지표 4종

| 지표 | 확인하는 것 |
|---|---|
| ① 픽셀 강도 분포 | 밝기 분포가 실제와 비슷한가 |
| ② 클래스 평균 상관 | DF 평균 형태를 재현했는가(높을수록 좋음) |
| ③ 최근접 실제 거리 | 0에 가까우면 학습 이미지를 베낀 것 |
| ④ 샘플 간 거리 | 0에 가까우면 mode collapse |

모두 `to_unit_scale`로 0 ~ 1로 맞춘 뒤 계산함(`practice/gen_metrics.py`).

In [ ]:
# 64장으로 평가 (8 ~ 16장이면 상관계수가 0.05 ~ 0.17까지 흔들림). 그림은 앞 4장
N_EVAL = 64
with torch.no_grad():  # 평가만 하므로 기울기 계산 생략
    fake = gen(N_EVAL).numpy().reshape(N_EVAL, IMG_SIZE, IMG_SIZE)

fake_u = gm.to_unit_scale(fake, "prob")   # 확률 척도 → 이미지별 min-max로 0~1
real_u = gm.to_unit_scale(df_full, "unit")  # 이미 0~1 - 그대로

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for j in range(4):
    axes[0, j].imshow(fake_u[j], cmap="gray"); axes[0, j].set_title(f"생성 {j}", fontsize=9); axes[0, j].axis("off")
    axes[1, j].imshow(real_u[j], cmap="gray"); axes[1, j].set_title(f"실제 DF {j}", fontsize=9); axes[1, j].axis("off")
plt.suptitle("QGAN 생성 이미지(위) vs 실제 DF(아래)")
plt.tight_layout(); plt.show()

In [ ]:
# 지표 ①② - 픽셀 분포 + 클래스 평균 상관계수
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
gm.plot_pixel_histogram(real_u, fake_u, ax=axes[0], label_real="실제 DF", label_fake="QGAN 생성")
axes[0].set_title("픽셀 강도 분포")

# 클래스 평균 이미지(전형적 형태)와의 상관 비교
df_mean = gm.class_mean_image(real_u)
nv_mean = gm.class_mean_image(gm.to_unit_scale(nv_full, "unit"))
corr_df = gm.mean_image_correlation(fake_u, df_mean)
corr_nv = gm.mean_image_correlation(fake_u, nv_mean)

axes[1].bar(["DF 평균 대비", "NV 평균 대비"], [corr_df.mean(), corr_nv.mean()])
axes[1].axhline(0, color="gray", lw=0.8); axes[1].set_ylabel("상관계수")
axes[1].set_title("클래스 평균 이미지와의 상관계수")
plt.tight_layout(); plt.show()

print(f"DF 평균 대비 상관계수: {corr_df.mean():.3f} (샘플별 최댓값 {corr_df.max():.3f})")
print(f"NV 평균 대비 상관계수: {corr_nv.mean():.3f}")
print("DF 쪽이 높아야 DF 특징을 학습한 것임")

In [ ]:
# 지표 ③ - 최근접 실제 이미지(암기 여부 확인)
fig, dist = gm.plot_nearest_neighbors(fake_u, real_u, IMG_SIZE, n_show=4)
plt.show()
print(f"최근접 실제 이미지까지의 L2 거리 - 평균 {dist.mean():.3f}, 최솟값 {dist.min():.3f}")
print("0에 가까우면 학습 데이터 암기")

# 지표 ④ - 샘플 간 다양성
div = gm.sample_diversity(fake_u)
print(f"\n샘플 간 거리 평균 {div['pairwise_mean']:.3f} / 최솟값 {div['pairwise_min']:.3f} / "
      f"픽셀별 표준편차 {div['pixelwise_std']:.3f}")
print("값이 0에 가까우면 mode collapse임")

In [ ]:
# 지표 4종 요약표 (3일차에서 MediQ-GAN과 비교)
table = gm.comparison_table({"QGAN(2일차 학습)": (fake, "prob")}, df_full)
print("=== 생성품질 지표 요약 ===")
display(table) if "display" in dir() else print(table)

## 7. 정리 - Drive에 저장

학습한 checkpoint와 직접 전처리한 샘플을 Google Drive에 저장함(원본 6GB는 저장하지 않음). 3일차는 저장소 자산으로 진행하므로 이 사본은 선택 사항임.

권한 창이 뜨면 본인 계정을 선택하고 **`모두 선택` → `계속`**. 일부만 허용하면 `drive.mount`가 실패하므로 셀을 다시 실행함.

<img src="https://raw.githubusercontent.com/hyunhp/skin_disease_qml/main/practice/docs/image/colab_drive_permission.png" width="640">

In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # 권한 창에서 모두 선택 → 계속 (위 캡처 참고)

save_dir = "/content/drive/MyDrive/QML강의"
os.makedirs(save_dir, exist_ok=True)

# 가중치 + 설정 + 손실 기록을 한 파일로 저장
torch.save({
    "generator_state_dict": gen.state_dict(),
    "config": {"n_generators": N_GENERATORS, "n_data_qubits": N_DATA_QUBITS,
               "n_ancillas": N_ANCILLA, "q_depth": Q_DEPTH, "img_size": IMG_SIZE,
               "target_class": "df", "scale": "prob", "steps": STEPS},
    "history": {"loss_g": hist_g, "loss_d": hist_d},
}, f"{save_dir}/qgan_my_{IMG_SIZE}px.pt")
np.save(f"{save_dir}/df_demo_{IMG_SIZE}px.npy", df_demo)
print(f"저장 완료: {save_dir}")
print("3일차는 저장소 자산으로 진행함 (이 파일로 대신해도 됨)")